# Fase 3 — Programación Orientada a Objetos

**Asignatura:** MCDI500 — Introducción a la Programación  
**Proyecto:** Detección de Anomalías y Posibles Fraudes en Permisos de Circulación Vehicular  
**Stack:** Python 3.14 · pandas · scikit-learn · loguru

# Proyecto de Magíster: Detección de Anomalías y Posibles Fraudes en Permisos de Circulación

## Problemática
En la administración municipal, el cobro de los Permisos de Circulación Vehicular depende fuertemente de datos ingresados manualmente y tasaciones. Debido a la falta de validación estandarizada en los sistemas locales, existen múltiples errores de digitación, inconsistencias de formato y, potencialmente, **fraudes y evasiones** (por ejemplo, vehículos de alto valor comercial registrados con características alteradas para pagar menos, o modificaciones vehiculares no declaradas). Esta desestructuración de la información genera ineficiencias en la gestión pública y grandes pérdidas de ingresos municipales.

## Objetivo General
Diseñar e implementar soluciones algorítmicas en Python aplicadas al proyecto transversal, integrando principios de programación estructurada, recursiva y orientada a objetos para construir código modular, reutilizable y escalable.

## Configuración del Entorno

Importación de librerías externas e internas. `loguru` reemplaza el módulo `logging` estándar entregando timestamps, niveles de color y formato estructurado sin configuración adicional.

In [ ]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd

from loguru import logger
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

## Arquitectura de Clases


Cada clase tiene **una sola responsabilidad** (alta cohesión) y expone un único método público orquestador que ejecuta los pasos privados en secuencia (bajo acoplamiento entre módulos).

### Clase `Explorador`

Analiza el dataset en crudo: dimensiones, tipos de datos, nulos, duplicados, tipos mixtos, anomalías de texto y coherencia de fechas. Todos los análisis son métodos privados (`_`); `explorar()` los ejecuta en orden y es el único punto de entrada público.

In [ ]:
class Explorador:
    """Clase para realizar un análisis exploratorio inicial del dataset,
    enfocándose en la estructura, calidad y posibles anomalías.
    Cada método privado se encarga de un aspecto específico del análisis,
    y el método público 'explorar' los ejecuta en secuencia."""

    def __init__(self, df):
        self.df = df

    def _find_col(self, *names):
        """Retorna el primer nombre de columna que exista en el df,
        usando comparación case-insensitive. Funciona antes y después de codificar()."""
        lower_map = {c.lower(): c for c in self.df.columns}
        for name in names:
            found = lower_map.get(name.lower())
            if found:
                return found
        return None

    def _explorar_estructura(self):
        try:
            logger.info("Estructura del dataset")
            print(f"Dimensiones: {self.df.shape[0]} filas x {self.df.shape[1]} columnas\n")
            display(self.df.head(10))
            print("\n--- Tipos de datos ---")
            display(self.df.dtypes.rename('tipo').to_frame())
            print("\n--- Estadísticas descriptivas ---")
            display(self.df.describe())
        except Exception as e:
            logger.error(f"_explorar_estructura falló: {e}")

    def _analizar_categorias(self, columnas=None):
        try:
            logger.info("Frecuencias por columna categórica")
            if columnas is None:
                columnas = self.df.select_dtypes(include='object').columns.tolist()
            for col in columnas:
                print(f"\n{col}:")
                print(self.df[col].value_counts(dropna=False))
                print("-" * 30)
        except Exception as e:
            logger.error(f"_analizar_categorias falló: {e}")

    def _generar_reporte_nulos(self):
        try:
            logger.info("Reporte de valores nulos")
            reporte = []
            for col in self.df.columns:
                nulos = self.df[col].isnull().sum()
                reporte.append({
                    'Columna': col,
                    'Tipo': str(self.df[col].dtype),
                    'Nulos': nulos,
                    '% Nulos': f"{nulos / len(self.df) * 100:.2f}%"
                })
            df_reporte = pd.DataFrame(reporte)
            con_nulos = df_reporte[df_reporte['Nulos'] > 0].sort_values('Nulos', ascending=False)
            display(con_nulos)
            return df_reporte
        except Exception as e:
            logger.error(f"_generar_reporte_nulos falló: {e}")

    def _detectar_duplicados(self):
        try:
            n = self.df.duplicated().sum()
            if n > 0:
                logger.warning(f"Filas totalmente duplicadas: {n}")
                col_id = self._find_col('_id', 'id')
                df_dup = self.df[self.df.duplicated(keep=False)]
                display(df_dup.sort_values(col_id).head(10) if col_id else df_dup.head(10))
            else:
                logger.success("Sin filas duplicadas")
            return n
        except Exception as e:
            logger.error(f"_detectar_duplicados falló: {e}")

    def _detectar_tipos_mixtos(self, columnas_numericas):
        try:
            logger.info("Detección de tipos mixtos en columnas numéricas")
            for col in [c for c in columnas_numericas if c in self.df.columns]:
                temp = pd.to_numeric(self.df[col], errors='coerce')
                mascara = self.df[col].notnull() & temp.isnull()
                if mascara.any():
                    logger.warning(f"'{col}': {self.df[mascara][col].unique()} ({mascara.sum()} filas con valores no numéricos)")
                else:
                    logger.success(f"'{col}': OK")
        except Exception as e:
            logger.error(f"_detectar_tipos_mixtos falló: {e}")

    def _analizar_anomalias_texto(self):
        try:
            logger.info("Longitud máxima en columnas de texto")
            cols_texto = [self._find_col(c) for c in ['TipoVehiculo', 'Marca', 'Modelo']]
            cols_texto = [c for c in cols_texto if c]
            for col in cols_texto:
                max_len = self.df[col].astype(str).map(len).max()
                if max_len > 50:
                    logger.warning(f"'{col}': {max_len} caracteres (supera umbral)")
                    display(self.df[self.df[col].astype(str).map(len) > 50][[col]].head())
                else:
                    logger.success(f"'{col}': {max_len} caracteres")
            col_tipo = self._find_col('TipoVehiculo')
            col_cil  = self._find_col('Cilindrada')
            if col_tipo and col_cil:
                logger.info(f"Cilindrada promedio por {col_tipo}")
                resumen = self.df.groupby(col_tipo)[col_cil].agg(['mean', 'max', 'min'])
                display(resumen.sort_values('mean', ascending=False))
        except Exception as e:
            logger.error(f"_analizar_anomalias_texto falló: {e}")


    def _detectar_candidatas_consolidacion(self, columnas=None):
        """Muestra distribución de cada columna categórica para identificar alias.

        Señala valores con frecuencia <1% del total — candidatos a ser consolidados
        con un valor más frecuente (ej. MOTO1/MOTO2 → MOTO, SUV 2 → STATION WAGON).
        """
        try:
            if columnas is None:
                columnas = self.df.select_dtypes(include='object').columns.tolist()
            for col in [c for c in columnas if c in self.df.columns]:
                vc = self.df[col].value_counts(dropna=False)
                if len(vc) <= 1:
                    continue
                print(f"\n{'─' * 45}")
                print(f"  {col}  ({len(vc)} categorías únicas)")
                print(f"{'─' * 45}")
                display(vc.to_frame('frecuencia').assign(
                    pct=lambda d: (d['frecuencia'] / d['frecuencia'].sum() * 100).round(2)
                ))
                umbral = vc.sum() * 0.01
                posibles_alias = vc[vc < umbral].index.tolist()
                if posibles_alias:
                    logger.warning(f"'{col}' — posibles alias (<1 %): {posibles_alias}")
                else:
                    logger.success(f"'{col}' — sin alias evidentes")
        except Exception as e:
            logger.error(f"_detectar_candidatas_consolidacion falló: {e}")

    def explorar(self, columnas_numericas=None):

        try:
            if columnas_numericas is None:
                columnas_numericas = self.df.select_dtypes(include='number').columns.tolist()
            self._explorar_estructura()
            self._analizar_categorias()
            self._generar_reporte_nulos()
            self._detectar_duplicados()
            self._detectar_tipos_mixtos(columnas_numericas)
            self._analizar_anomalias_texto()
            logger.success("Exploración completada")
        except Exception as e:
            logger.error(f"explorar falló: {e}")

### Jerarquía de Transformadores: Herencia y Polimorfismo


`Transformador` es una clase abstracta (`ABC`) que define el contrato: toda subclase debe implementar `aplicar(df) → DataFrame`. Esto garantiza una interfaz uniforme sin importar la estrategia concreta.

```
Transformador  (ABC — contrato)
├── ImputarModa          → rellena con la moda de la columna
├── ImputarMedia         → rellena con la media aritmética
├── ImputarMediana       → rellena con mediana, opcionalmente agrupando por categoría
├── ImputarValorFijo     → rellena con un valor literal definido por el usuario
├── EscalarZScore        → StandardScaler  (media=0, std=1)
└── EscalarMinMax        → MinMaxScaler    (rango [0, 1])
```

El `Preprocesador` llama `.imputar(columna, metodo)` por cada columna — el método despacha al `Transformador` correspondiente sin conocer la implementación concreta — ese es el **polimorfismo**.

In [ ]:
from abc import ABC, abstractmethod
from sklearn.preprocessing import StandardScaler, MinMaxScaler


class Transformador(ABC):
    """Contrato común para todos los transformadores del pipeline.

    Cualquier clase que herede debe implementar aplicar(self, df)
    y retornar el DataFrame modificado.
    """

    @abstractmethod
    def aplicar(self, df):
        """Aplica la transformación sobre df y lo retorna."""
        ...


# ── Imputación ───────────────────────────────────────────────────────────────

class ImputarModa(Transformador):
    """Imputa nulos de columnas categóricas con la moda."""

    def __init__(self, columnas):
        self.columnas = columnas if isinstance(columnas, list) else [columnas]

    def aplicar(self, df):
        try:
            for col in self.columnas:
                if col in df.columns:
                    df[col] = df[col].fillna(df[col].mode()[0])
                    logger.info(f"ImputarModa: '{col}' imputada con moda")
        except Exception as e:
            logger.error(f"ImputarModa.aplicar falló: {e}")
        return df


class ImputarMedia(Transformador):
    """Imputa nulos numéricos con la media aritmética."""

    def __init__(self, columna):
        self.columna = columna

    def aplicar(self, df):
        try:
            df[self.columna] = df[self.columna].fillna(df[self.columna].mean())
            logger.info(f"ImputarMedia: '{self.columna}' imputada con media")
        except Exception as e:
            logger.error(f"ImputarMedia.aplicar falló: {e}")
        return df


class ImputarMediana(Transformador):
    """Imputa nulos numéricos con la mediana, opcionalmente agrupando por otra columna."""

    def __init__(self, columna, grupo=None):
        self.columna = columna
        self.grupo   = grupo

    def aplicar(self, df):
        try:
            if self.grupo and self.grupo in df.columns:
                df[self.columna] = (
                    df.groupby(self.grupo)[self.columna]
                    .transform(lambda x: x.fillna(x.median()))
                )
            df[self.columna] = df[self.columna].fillna(df[self.columna].median())
            logger.info(f"ImputarMediana: '{self.columna}' imputada con mediana")
        except Exception as e:
            logger.error(f"ImputarMediana.aplicar falló: {e}")
        return df


class ImputarValorFijo(Transformador):
    """Imputa nulos con un valor fijo por columna."""

    def __init__(self, columnas_valores: dict):
        self.columnas_valores = columnas_valores

    def aplicar(self, df):
        try:
            for col, valor in self.columnas_valores.items():
                if col in df.columns:
                    df[col] = df[col].fillna(valor)
                    logger.info(f"ImputarValorFijo: '{col}' → '{valor}'")
        except Exception as e:
            logger.error(f"ImputarValorFijo.aplicar falló: {e}")
        return df


class ConsolidarCategorias(Transformador):
    """Reemplaza alias en una columna categórica mediante un mapeo."""

    def __init__(self, columna: str, mapeo: dict):
        self.columna = columna
        self.mapeo   = mapeo

    def aplicar(self, df):
        try:
            if self.columna in df.columns:
                df[self.columna] = df[self.columna].replace(self.mapeo)
                logger.info(f"ConsolidarCategorias: '{self.columna}' consolidada")
        except Exception as e:
            logger.error(f"ConsolidarCategorias.aplicar falló: {e}")
        return df


# ── Escalado ─────────────────────────────────────────────────────────────────

class EscalarZScore(Transformador):
    """Escala columnas numéricas con z-score (media=0, std=1)."""

    def __init__(self, columnas):
        self.columnas = columnas
        self.scaler   = StandardScaler()

    def aplicar(self, df):
        try:
            cols = [c for c in self.columnas if c in df.columns]
            df[cols] = self.scaler.fit_transform(df[cols])
            logger.info(f"EscalarZScore aplicado a: {cols}")
        except Exception as e:
            logger.error(f"EscalarZScore.aplicar falló: {e}")
        return df


class EscalarMinMax(Transformador):
    """Escala columnas numéricas al rango [0, 1]."""

    def __init__(self, columnas):
        self.columnas = columnas
        self.scaler   = MinMaxScaler()

    def aplicar(self, df):
        try:
            cols = [c for c in self.columnas if c in df.columns]
            df[cols] = self.scaler.fit_transform(df[cols])
            logger.info(f"EscalarMinMax aplicado a: {cols}")
        except Exception as e:
            logger.error(f"EscalarMinMax.aplicar falló: {e}")
        return df

### Clase `Limpiador`

Responsabilidad única: **limpieza estructural del DataFrame**. Encapsula las dos operaciones que preparan el dato crudo antes de imputar:

| Método | Responsabilidad |
|---|---|
| `_eliminar_columnas(columnas)` | Elimina columnas sin aporte analítico |
| `_convertir_numericas(columnas)` | Fuerza dtype numérico; valores no convertibles → NaN para imputar después |

`Preprocesador` delega con `Limpiador(self.df)._eliminar_columnas(...)` y `Limpiador(self.df)._convertir_numericas(...)` sin conocer los detalles internos.

In [ ]:
class Limpiador:
    """Limpieza estructural: elimina columnas y fuerza tipos numéricos."""

    def __init__(self, df):
        self.df = df

    def _eliminar_columnas(self, columnas):
        for col in columnas:
            if col in self.df.columns:
                self.df = self.df.drop(columns=[col])
                logger.info(f"Columna '{col}' eliminada")
        logger.success(f"Limpieza — {self.df.shape[0]} filas x {self.df.shape[1]} columnas")
        return self.df

    def _convertir_numericas(self):
        """Detecta columnas object que contienen datos numéricos y las convierte.

        Criterio: >80% de los valores no nulos se convierten a número sin error.
        """
        candidatas = [
            col for col in self.df.select_dtypes(include='object').columns
            if pd.to_numeric(self.df[col], errors='coerce').notna().mean() > 0.8
        ]
        for col in candidatas:
            antes = self.df[col].isna().sum()
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
            convertidos = self.df[col].isna().sum() - antes
            if convertidos > 0:
                logger.warning(f"'{col}': {convertidos} valor(es) no numérico(s) convertidos a NaN")
            else:
                logger.info(f"'{col}': convertida a numérico sin errores")
        logger.success(f"Conversión numérica completada — {len(candidatas)} columnas: {candidatas}")
        return self.df


### Clase `Codificador`

Responsabilidad única: **normalización y codificación**. Centraliza las transformaciones de texto (quitar tildes, mayúsculas), fechas y el renombrado de columnas a `snake_case`.

| Método | Responsabilidad |
|---|---|
| `_sin_tildes(texto)` | Elimina diacríticos vía `unicodedata.normalize` |
| `_a_snake_case(nombre)` | Convierte `CamelCase` y espacios a `snake_case` |
| `codificar()` | Orquestador público — normaliza texto, fechas y columnas |

El `Preprocesador` delega con `Codificador(self.df).codificar()` sin conocer los detalles internos.

In [ ]:
class Codificador:
    """Normaliza texto, fechas y renombra columnas a snake_case.

    Detecta automáticamente las columnas de texto (dtype object) del df.
    """

    def __init__(self, df):
        self.df = df.copy()

    @staticmethod
    def _sin_tildes(texto):
        if not isinstance(texto, str):
            return texto
        return "".join(
            c for c in unicodedata.normalize('NFD', texto)
            if not unicodedata.combining(c)
        )

    @staticmethod
    def _a_snake_case(nombre):
        s = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', nombre)
        s = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s).lower()
        return s.replace(' ', '_').replace('.', '').replace('__', '_').strip('_')

    def codificar(self):
        try:
            cols_texto = self.df.select_dtypes(include='object').columns.tolist()
            for col in cols_texto:
                self.df[col] = self.df[col].str.strip().str.upper().apply(self._sin_tildes)
            logger.info(f"Texto normalizado — {len(cols_texto)} columnas")

            for col in ['Fecha_Emision', 'Fecha_Vencimiento']:
                if col in self.df.columns:
                    self.df[col] = pd.to_datetime(self.df[col], errors='coerce').dt.date
            logger.info("Fechas normalizadas")

            self.df.columns = [self._a_snake_case(col) for col in self.df.columns]
            logger.success(f"Codificación completada — columnas: {self.df.columns.tolist()}")
        except Exception as e:
            logger.error(f"codificar falló: {e}")
        return self.df

### Suite de Pruebas


Prueba exhaustiva del método `validar()` del `Preprocesador` cubriendo tres categorías:

| Tipo | Qué verifica |
|---|---|
| **Normal** | Años y valores dentro de rangos esperados — debe pasar sin error |
| **Límite** | Valores en los bordes del rango válido — comportamiento en el extremo |
| **Excepción** | Entradas inválidas — debe lanzar `AssertionError` |

Cada caso usa `assert` para verificar el resultado esperado y reporta `[PASS]` / `[FAIL]` con `loguru`.

In [ ]:
class TestValidaciones:
    """Suite de pruebas para validar que el método validar() de Preprocesador
    detecta correctamente casos normales, límite y de excepción."""

    @staticmethod
    def _chk(cond, msg):
        if not cond:
            raise AssertionError(msg)

    @staticmethod
    def _caso(nombre, fn):
        try:
            fn()
            logger.success(f"[PASS] {nombre}")
        except AssertionError as e:
            logger.warning(f"[FAIL] {nombre} → {e}")
        except Exception as e:
            logger.error(f"[ERROR] {nombre} → {e}")

    def ejecutar(self, anio_actual=2026):
        logger.info("── Iniciando suite de pruebas de validar ──")

        # ── Caso normal ───────────────────────────────────────────────────
        df_ok = pd.DataFrame({
            'ano_fabricacion': [2000, 2010, 2020],
            'cilindrada':      [1600, 2000, 1400],
        })
        self._caso(
            "Normal — sin nulos, sin duplicados, años y cilindradas válidos",
            lambda: (
                self._chk(df_ok.isna().sum().sum() == 0,                             "nulos inesperados"),
                self._chk(not df_ok.duplicated().any(),                              "duplicados inesperados"),
                self._chk(df_ok['ano_fabricacion'].between(1900, anio_actual).all(), "año fuera de rango"),
                self._chk((df_ok['cilindrada'] >= 0).all(),                         "cilindrada negativa"),
            )
        )

        # ── Casos límite ──────────────────────────────────────────────────
        df_limite = pd.DataFrame({
            'ano_fabricacion': [1900, anio_actual],
            'cilindrada':      [0, 9999],
        })
        self._caso(
            f"Límite — ano_fabricacion=[1900, {anio_actual}], cilindrada=[0, 9999]",
            lambda: (
                self._chk(df_limite['ano_fabricacion'].between(1900, anio_actual).all(), "año fuera de rango"),
                self._chk((df_limite['cilindrada'] >= 0).all(),                          "cilindrada negativa"),
            )
        )

        # ── Casos de excepción ────────────────────────────────────────────
        df_nulos = pd.DataFrame({'ano_fabricacion': [2000, None], 'cilindrada': [1600, 1400]})
        self._caso(
            "Excepción — nulos presentes (debe fallar)",
            lambda: self._chk(df_nulos.isna().sum().sum() == 0, "Existen valores nulos")
        )

        df_duplicados = pd.DataFrame({'ano_fabricacion': [2000, 2000], 'cilindrada': [1600, 1600]})
        self._caso(
            "Excepción — filas duplicadas (debe fallar)",
            lambda: self._chk(not df_duplicados.duplicated().any(), "Existen filas duplicadas")
        )

        df_anio_fuera = pd.DataFrame({'ano_fabricacion': [1800, 2050], 'cilindrada': [1600, 1400]})
        self._caso(
            f"Excepción — años fuera de rango [1800, 2050] (debe fallar)",
            lambda: self._chk(df_anio_fuera['ano_fabricacion'].between(1900, anio_actual).all(),
                              "Años de fabricación fuera de rango")
        )

        df_cilindrada_neg = pd.DataFrame({'ano_fabricacion': [2000, 2010], 'cilindrada': [1600, -100]})
        self._caso(
            "Excepción — cilindrada negativa (debe fallar)",
            lambda: self._chk((df_cilindrada_neg['cilindrada'] >= 0).all(), "Cilindradas negativas detectadas")
        )

        logger.info("── Suite de pruebas finalizada ──")

### Clase `Preprocesador` — Orquestador del Pipeline

Coordina todas las clases anteriores en un **pipeline fluido** mediante encadenamiento de métodos (*method chaining*): cada método retorna `self`, permitiendo llamadas consecutivas sobre el mismo objeto sin variables intermedias.

```python
Preprocesador(ruta)
    .cargar_datos()
    .limpiar()          # elimina columnas, convierte tipos
    .imputar()          # ImputarModa / Mediana / ValorFijo
    .consolidar()       # ConsolidarCategorias
    .codificar()        # normaliza texto, fechas y snake_case
    .escalar([...])     # usa EscalarZScore / EscalarMinMax
    .validar()          # misma lógica que TestValidaciones
    .exportar(ruta)
    .resultado()        # retorna el DataFrame final
```

In [ ]:
class Preprocesador:
    """Objeto que guarda una tabla de datos y sabe prepararla para el analisis.

    Cada metodo del pipeline retorna self para permitir encadenamiento fluido:
        df = Preprocesador(ruta).cargar_datos().limpiar().imputar(...).consolidar().codificar().resultado()
    """

    def __init__(self, ruta):
        self.ruta   = ruta
        self.df     = None
        self.scaler = None

    def cargar_datos(self):
        """Carga el dataset desde disco y lo guarda en self.df."""
        try:
            if not os.path.exists(self.ruta):
                raise FileNotFoundError(f"No se encontró el archivo: '{self.ruta}'")
            self.df = pd.read_csv(self.ruta, encoding='utf-8')
            logger.success(f"Dataset cargado — {self.df.shape[0]} filas x {self.df.shape[1]} columnas")
            return self
        except Exception as e:
            logger.error(f"cargar_datos falló: {e}")
            return self

    def explorar(self, columnas_numericas=None):
        """Análisis exploratorio inicial del dataset."""
        try:
            Explorador(self.df).explorar(columnas_numericas)
        except Exception as e:
            logger.error(f"explorar falló: {e}")
        return self

    def limpiar(self, columnas):
        """Elimina columnas del DataFrame delegando a Limpiador."""
        try:
            self.df = Limpiador(self.df)._eliminar_columnas(columnas)
            self.df = Limpiador(self.df)._convertir_numericas()
        except Exception as e:
            logger.error(f"limpiar falló: {e}")
        return self

    def imputar(self, columna, metodo, valor=None):
        """Imputa nulos de una columna via Transformadores según el método elegido.

        metodo='moda'       → ImputarModa
        metodo='media'      → ImputarMedia
        metodo='mediana'    → ImputarMediana
        metodo='valor_fijo' → ImputarValorFijo  (requiere valor=str)
        """
        try:
            metodos = {
                'moda':       ImputarModa(columna),
                'media':      ImputarMedia(columna),
                'mediana':    ImputarMediana(columna),
                'valor_fijo': ImputarValorFijo({columna: valor}),
            }
            if metodo not in metodos:
                raise ValueError(f"Método desconocido: '{metodo}'. Usa moda, media, mediana o valor_fijo.")
            self.df = metodos[metodo].aplicar(self.df)
            logger.success(f"Imputación '{columna}' con {metodo} completada")
        except Exception as e:
            logger.error(f"imputar falló: {e}")
        return self

    def consolidar(self, transformadores):
        """Consolida alias categóricos aplicando cada ConsolidarCategorias recibido.

        Los mapeos se derivan del análisis exploratorio (_analizar_categorias)
        y se pasan desde el pipeline — el Preprocesador no asume qué consolidar.
        """
        try:
            for t in transformadores:
                self.df = t.aplicar(self.df)
            logger.success("Categorías consolidadas")
        except Exception as e:
            logger.error(f"consolidar falló: {e}")
        return self

    def codificar(self):
        """Delega la normalización de texto, fechas y snake_case a Codificador."""
        try:
            self.df = Codificador(self.df).codificar()
        except Exception as e:
            logger.error(f"codificar falló: {e}")
        return self

    def escalar(self, columnas, metodo='zscore'):
        """Escala columnas numéricas usando un Transformador según el método elegido.

        metodo='zscore'  → EscalarZScore  (media=0, std=1)
        metodo='minmax'  → EscalarMinMax  (rango [0, 1])
        """
        try:
            transformadores = {
                'zscore': EscalarZScore(columnas),
                'minmax': EscalarMinMax(columnas),
            }
            if metodo not in transformadores:
                raise ValueError(f"Método desconocido: '{metodo}'. Usa 'zscore' o 'minmax'.")
            self.df = transformadores[metodo].aplicar(self.df)
        except Exception as e:
            logger.error(f"escalar falló: {e}")
        return self

    def validar(self, anio_actual=2026):
        """Verifica condiciones mínimas de calidad del dataset antes de exportar."""
        TestValidaciones().ejecutar(anio_actual)

        logger.info("── Validando dataset real ──")
        try:
            assert self.df.isna().sum().sum() == 0,                             "Existen valores nulos"
            assert not self.df.duplicated().any(),                              "Existen filas duplicadas"
            assert self.df['ano_fabricacion'].between(1900, anio_actual).all(), "Años de fabricación fuera de rango"
            assert (self.df['cilindrada'] >= 0).all(),                          "Cilindradas negativas detectadas"
            logger.success("Dataset real: todas las validaciones pasaron")
        except AssertionError as e:
            logger.error(f"validar falló: {e}")
        except Exception as e:
            logger.error(f"validar — error inesperado: {e}")
        return self

    def exportar(self, ruta):
        """Guarda el dataset procesado en disco."""
        try:
            os.makedirs(os.path.dirname(ruta), exist_ok=True)
            self.df.to_csv(ruta, index=False, encoding='utf-8')
            kb = os.path.getsize(ruta) / 1024
            logger.success(f"Dataset exportado → {ruta}  ({kb:.1f} KB | {self.df.shape[0]} filas x {self.df.shape[1]} columnas)")
        except Exception as e:
            logger.error(f"exportar falló: {e}")
        return self

    def resultado(self):
        """Retorna el DataFrame procesado al final del pipeline."""
        return self.df

### Documentación de Arquitectura


La arquitectura sigue el principio de **una responsabilidad por componente**. La tabla resume cada pieza y el patrón de diseño que la guía:

| Componente | Responsabilidad | Patrón aplicado |
|---|---|---|
| `Explorador` | Análisis exploratorio del dataset | Encapsulamiento de estado |
| `Limpiador` | Limpieza, casting e imputación | Facade (delega a `Transformadores`) |
| `Transformador` (ABC) | Contrato común de transformación | Template Method |
| `ImputarModa / Mediana / ValorFijo` | Estrategias de imputación intercambiables | Strategy |
| `EscalarZScore / MinMax` | Estrategias de escalado intercambiables | Strategy |
| `TestValidaciones` | Verificación con casos normal / límite / excepción | Test Suite |
| `Preprocesador` | Orquestador del pipeline completo | Fluent Interface (*method chaining*) |

**Decisiones técnicas y su justificación:**

- **`_` en atributos y métodos**: protege el estado interno; solo los métodos públicos son la interfaz — *encapsulamiento*
- **`@abstractmethod` en `Transformador`**: garantiza que toda subclase implemente `aplicar(df)` — el error de diseño se detecta al instanciar, no en tiempo de ejecución
- **`return self`**: habilita el *method chaining* sin variables intermedias; hace el pipeline legible como secuencia de pasos
- **`loguru`** en lugar de `print` o `logging`: contexto automático (timestamp, nivel, módulo) sin boilerplate en cada clase; `print` se reserva para salidas visuales/tabulares

## Preprocesamiento y Ejecución del Pipeline


El pipeline ejecuta el preprocesamiento completo en cuatro pasos: carga y limpieza del dataset, escalado de variables numéricas, validación de coherencia y exportación del resultado. Cada paso corre en celda separada para inspeccionar los logs de forma independiente.

La variable `pipe` mantiene el estado entre celdas gracias a que cada método retorna `self` (*Fluent Interface*). Todas las decisiones de transformación quedan trazables en los logs de `loguru`.

### Paso 1 — Carga, Limpieza e Imputación

Ejecuta la cadena completa de preparación en orden:

1. **`limpiar(columnas)`** — elimina columnas sin aporte analítico
2. **`imputar()`** — rellena nulos con moda, mediana o valor fijo vía `Transformadores`
3. **`consolidar()`** — unifica aliases categóricos (`SEDAN→AUTOMOVIL`, `CVT→AUT`, ...) vía `ConsolidarCategorias`
4. **`codificar()`** — normaliza texto (mayúsculas, sin tildes), fechas y renombra columnas a `snake_case`

In [ ]:
# Paso 1: limpiar → imputar → consolidar → codificar
pipe = (Preprocesador("../datos/original/permiso-circulacion-2026.csv")
        .cargar_datos()
        .explorar())

#### Limpiar: criterio de eliminación de columnas

`limpiar()` delega en `Limpiador` dos operaciones internas antes de pasar al siguiente paso:

| Operación interna | Qué hace |
|---|---|
| `_eliminar_columnas(columnas)` | Descarta columnas sin aporte analítico |
| `_convertir_numericas()` | Detecta columnas `object` con >80 % de valores numéricos y fuerza su dtype — los no convertibles pasan a `NaN` para imputar después |

Columnas eliminadas y su motivo:

| Columna | Motivo |
|---|---|
| `Equipamiento` | >55 % de nulos; los valores presentes son el literal `'EQUI'` sin definición de dominio |
| `Tonelaje` | >98 % con valor `0` — sin variabilidad, no discrimina entre vehículos |

`codificar()` se encadena al final del mismo paso para normalizar texto y renombrar columnas a `snake_case` antes de imputar.

In [ ]:
pipe = (pipe
        .limpiar([
            'Equipamiento',  # >55% sin información útil (nulos + 'EQUI' sin definición)
            'Tonelaje',      # >98% de los registros tienen valor 0, sin aporte analítico
        ])
        .codificar()) 


pipe.df.head(10)

#### Consolidar: unificación de alias categóricos

### Análisis previo: candidatas a consolidación

Antes de definir los mapeos de `ConsolidarCategorias`, se inspeccionan las columnas
categóricas clave con `_detectar_candidatas_consolidacion`. Los valores con frecuencia
**<1 %** del total se marcan como posibles alias — esos son los que se consolidan.

In [ ]:
# Sin parámetros: escanea todas las columnas object y señala las que tienen alias
Explorador(pipe.df)._detectar_candidatas_consolidacion([
    'tipo_vehiculo', 'transmision', 'combustible'
])


`consolidar()` aplica `ConsolidarCategorias` sobre las columnas que el análisis exploratorio marcó con valores de frecuencia <1 % — variantes ortográficas o sub-categorías redundantes de una categoría mayor.

| Columna | Alias detectados | Categoría canónica |
|---|---|---|
| `tipo_vehiculo` | `SEDAN`, `SEDAN 2` → `AUTOMOVIL`; `SUV`, `SUV 2` → `STATION WAGON`; `MOTO1`, `MOTO2` → `MOTO`; `VAN 2` → `FURGON`; variantes `MINIBUS` → `MINIBUS` | unificados en 10 categorías |
| `transmision` | `CVT` → `AUT` | dos valores: `MEC` / `AUT` |
| `combustible` | `MEC` → `DIES` | cuatro valores: `DIES` / `BENC` / `HIBR` / `ELEC` |

El mapeo es explícito y trazable: no hay inferencia — solo sustitución de alias conocidos.

In [ ]:
pipe = (pipe
        .consolidar([
            # Mapeos derivados del análisis exploratorio (_analizar_categorias)
            ConsolidarCategorias('TipoVehiculo', {
                'SEDAN': 'AUTOMOVIL',      'SEDAN 2': 'AUTOMOVIL',
                'SUV': 'STATION WAGON',    'SUV 2': 'STATION WAGON',
                'MOTO1': 'MOTO',           'MOTO2': 'MOTO',
                'VAN 2': 'FURGON',
                'MINIBUS PARTICULAR':         'MINIBUS',
                'MINIBUS TRANS  PASAJERO':    'MINIBUS',
                'MINIBUS TRANS PASAJERO':     'MINIBUS',
                'MINIBUS DE TURISMO':         'MINIBUS',
                'MINIBUS ESCOLAR':            'MINIBUS',
            }),
            ConsolidarCategorias('Transmision', {'CVT': 'AUT'}),
            ConsolidarCategorias('Combustible', {'MEC': 'DIES'}),
        ])
)

In [ ]:
Explorador(pipe.df)._generar_reporte_nulos()

#### Imputar: motivos

| Columna | Nulos | % | Método | Justificación |
|---|---|---|---|---|
| `combustible` | 110 | 3.44 % | **moda** | Categórica nominal sin escala numérica. La moda (DIES, 54 %) es el único estadístico con sentido; media y mediana no aplican. |
| `transmision` | 122 | 3.82 % | **moda** | Categórica nominal. La moda (MEC, 69 %) representa el comportamiento más frecuente del parque vehicular. |
| `cilindrada` | 112 | 3.51 % | **mediana** | Numérica con sesgo a la derecha (50 cc–3 700+ cc). La mediana resiste los extremos; la media quedaría arrastrada por los valores altos de camiones. |
| `codigo_sii` | 367 | 11.49 % | **valor fijo `'SIN_CODIGO'`** | 1 677 valores únicos — la moda no predice nada. Además es un identificador: imputar con un código real equivaldría a asignarle el vehículo a otro propietario. Valor centinela obligatorio. |
| `comuna_anterior` | 12 | 0.38 % | **valor fijo `'OTRA'`** | Probable vehículo sin registro previo de traspaso. Asignar la moda (RIO IBAÑEZ, 76 %) sería semánticamente incorrecto; `'OTRA'` explicita la ausencia sin inventar un origen. |

In [ ]:
pipe = (pipe
        .imputar('combustible',     'moda')
        .imputar('transmision',     'moda')
        .imputar('cilindrada',      'mediana')
        .imputar('codigo_sii',      'valor_fijo', 'SIN_CODIGO')
        .imputar('comuna_anterior', 'valor_fijo', 'OTRA'))

Explorador(pipe.df)._generar_reporte_nulos()

### Paso 2 — Escalado de Variables Numéricas

Estandariza variables numéricas con dos estrategias vía polimorfismo:
- **z-score** (`EscalarZScore`): media=0, std=1 — para modelos que asumen distribución normal
- **min-max** (`EscalarMinMax`): rango [0,1] — para variables acotadas o redes neuronales

In [ ]:
# Paso 2: escalar variables numéricas
pipe = (pipe
        .escalar(['valor_contado', 'cilindrada', 'ano_fabricacion'])   # z-score
        .escalar(['total_a_pagar'], metodo='minmax'))                  # [0, 1]

### Paso 3 — Validación

Verifica coherencia interna del dataset: rangos de años, valores positivos y consistencia de fechas. Comparte la lógica con la suite `TestValidaciones` — si pasa aquí, pasa allá.

In [ ]:
# Paso 3: validar coherencia de los datos
pipe = pipe.validar()

### Paso 4 — Exportar y Resultado Final

Persiste el DataFrame procesado en `datos/resultado/df_clean.csv` y retorna el objeto `DataFrame` para continuar el análisis en celdas siguientes.

In [ ]:
# Paso 4: exportar CSV y extraer DataFrame final
df_listo = (pipe
            .exportar("../datos/resultado/df_clean.csv")
            .resultado())
df_listo.head()

## Eficiencia y Optimización


Mediciones reproducibles con `timeit` comparando dos pares de implementaciones. El análisis usa **Big-O** para distinguir complejidad teórica de rendimiento práctico: dos algoritmos O(n) pueden diferir por un orden de magnitud según el overhead de Python.

| Comparación | Implementación lenta | Implementación rápida | Diferencia |
|---|---|---|---|
| Normalización de texto | `iterrows` — Python puro | `.str` vectorizado — C/NumPy | 10–100× |
| Búsqueda de elemento | `list` — O(n) lineal | `set` — O(1) tabla hash | proporcional a n |

In [ ]:
import timeit
import pandas as pd

df_bench = df_listo.copy()


def normalizar_bucle(df):
    """Aplica strip+upper fila por fila con iterrows — O(n) lento."""
    resultado = []
    for _, fila in df.iterrows():
        resultado.append(fila['marca'].strip().upper() if isinstance(fila['marca'], str) else fila['marca'])
    return resultado

def normalizar_vectorizado(df):
    """Aplica strip+upper sobre toda la columna a la vez — O(n) rápido."""
    return df['marca'].str.strip().str.upper()

N = 5
t_bucle       = timeit.timeit(lambda: normalizar_bucle(df_bench),       number=N)
t_vectorizado = timeit.timeit(lambda: normalizar_vectorizado(df_bench), number=N)

# print: tabla comparativa con separadores y alineación — logger rompería el formato de columnas
print("─" * 55)
print("  COMPARACIÓN 1 — Normalización de texto (marca)")
print("─" * 55)
print(f"  iterrows (bucle fila a fila) : {t_bucle:.4f} s")
print(f"  vectorizado (.str)           : {t_vectorizado:.4f} s")
print(f"  Speedup                      : {t_bucle / t_vectorizado:.1f}x más rápido")
print()
print("  Big-O:")
print("    iterrows   → O(n)  — Python puro, overhead por fila")
print("    vectorizado → O(n) — misma complejidad teórica, pero ejecutado")
print("                         en C (NumPy/pandas), sin overhead de Python")
print("─" * 55)


# Pregunta: ¿existe 'TOYOTA' en el dataset?
# list recorre elemento a elemento; set usa tabla hash → acceso directo.

marcas_list = df_bench['marca'].dropna().tolist()
marcas_set  = set(marcas_list)

M = 10_000
t_list = timeit.timeit(lambda: 'TOYOTA' in marcas_list, number=M)
t_set  = timeit.timeit(lambda: 'TOYOTA' in marcas_set,  number=M)

print()
print("─" * 55)
print("  COMPARACIÓN 2 — Búsqueda de marca: list vs set")
print("─" * 55)
print(f"  list ({len(marcas_list):,} elementos) : {t_list:.4f} s")
print(f"  set  ({len(marcas_set):,} elementos únicos) : {t_set:.4f} s")
print(f"  Speedup                  : {t_list / t_set:.1f}x más rápido")
print()
print("  Big-O:")
print("    list → O(n) — recorre hasta encontrar el elemento")
print("    set  → O(1) — tabla hash, acceso en tiempo constante")
print("─" * 55)
logger.success(f"Normalización — speedup vectorizado: {t_bucle / t_vectorizado:.1f}x")
logger.success(f"Búsqueda      — speedup set vs list: {t_list / t_set:.1f}x")
logger.info("Para texto masivo usar .str vectorizado; para búsquedas repetidas usar set.")

## Diseño Estructurado y Recursividad


**Alta cohesión, bajo acoplamiento**: cada función hace exactamente una cosa.

| Función | Responsabilidad única |
|---|---|
| `_normalizar_texto` | quitar tildes y pasar a mayúsculas |
| `_cargar_mapa_comunas` | construir lookup `{COMUNA → región}` |
| `buscar_region` | **recursivo** — recorre la lista de regiones hasta encontrar la comuna |
| `normalizar_columnas` | **recursivo** — normaliza columnas de a una, pasa el resto a sí misma |

**Aplicación al proyecto**: enriquecer el dataset con `region_propietario` y `region_permiso`
a partir de `datos/comunas-regiones.json`, útil para detectar inconsistencias geográficas
(ej. vehículo con permiso en una región pero propietario registrado en otra).

In [ ]:
import json
import unicodedata


# ── Alta cohesión: cada función hace una sola cosa ─────────────────────────
def _normalizar_texto(texto: str) -> str:
    sin_tildes = unicodedata.normalize('NFD', str(texto)).encode('ascii', 'ignore').decode()
    return sin_tildes.upper().strip()


def _cargar_mapa_comunas(ruta: str) -> dict:
    try:
        alias = {
            'COYHAIQUE':           'Región Aisén del Gral. Carlos Ibáñez del Campo',
            'PUERTO NATALES':      'Región de Magallanes y de la Antártica Chilena',
            'PUERTO CISNES':       'Región Aisén del Gral. Carlos Ibáñez del Campo',
            'CON-CON':             'Valparaíso',
            "SAN VICENTE DE T.T.": "Región del Libertador Gral. Bernardo O'Higgins",
        }
        with open(ruta) as f:
            data = json.load(f)
        mapa = {
            _normalizar_texto(comuna): entrada['region']
            for entrada in data['regiones']
            for comuna in entrada['comunas']
        }
        mapa.update(alias)
        return mapa
    except Exception as e:
        logger.error(f"_cargar_mapa_comunas falló: {e}")
        return {}


# ── ALGORITMO RECURSIVO 1: búsqueda de región por índice ───────────────────
def buscar_region(regiones: list, comuna: str, idx: int = 0) -> str:
    if idx >= len(regiones):                              # CASO BASE: no encontrada
        return 'SIN_REGION'
    comunas_norm = [_normalizar_texto(c) for c in regiones[idx]['comunas']]
    if _normalizar_texto(comuna) in comunas_norm:         # CASO BASE: encontrada
        return regiones[idx]['region']
    return buscar_region(regiones, comuna, idx + 1)       # CASO RECURSIVO


# ── ALGORITMO RECURSIVO 2: normalizar lista de columnas ────────────────────
def normalizar_columnas(df, columnas: list):
    if not columnas:                                      # CASO BASE: lista vacía
        return df
    col = columnas[0]
    df[col] = df[col].str.upper().str.strip()
    return normalizar_columnas(df, columnas[1:])          # CASO RECURSIVO


# ── Aplicación al dataset ───────────────────────────────────────────────────
try:
    with open('../datos/comunas-regiones.json') as f:
        regiones_data = json.load(f)['regiones']

    mapa = _cargar_mapa_comunas('../datos/comunas-regiones.json')

    # Demo del algoritmo recursivo en casos puntuales
    print('Demo buscar_region():')
    print(f"  RENCA        → {buscar_region(regiones_data, 'RENCA')}")
    print(f"  TEMUCO       → {buscar_region(regiones_data, 'TEMUCO')}")
    print(f"  AISEN        → {buscar_region(regiones_data, 'AISEN')}")
    print(f"  DESCONOCIDA  → {buscar_region(regiones_data, 'DESCONOCIDA')}")

    df_listo['region_propietario'] = df_listo['comuna_propietario'].map(
        lambda c: mapa.get(_normalizar_texto(c), 'SIN_REGION')
    )
    df_listo['region_permiso'] = df_listo['comuna_permiso'].map(
        lambda c: mapa.get(_normalizar_texto(c), 'SIN_REGION')
    )

    df_listo = normalizar_columnas(df_listo, ['region_propietario', 'region_permiso'])
    logger.success("Enriquecimiento geográfico completado")
except Exception as e:
    logger.error(f"Enriquecimiento geográfico falló: {e}")

### Re-integración con el Pipeline

Tras el enriquecimiento, el DataFrame actualizado se sincroniza de vuelta al `Preprocesador` con `pipe.df = df_listo`. Esto permite reutilizar los métodos genéricos `exportar()` y `resultado()` ya definidos — sin duplicar lógica de escritura ni de logging.

Finalmente, `Explorador` revisa el dataset enriquecido con las dos columnas nuevas (`region_propietario`, `region_permiso`) para confirmar que la integración es correcta.

In [ ]:
# Sincronizar el df enriquecido de vuelta al Preprocesador
pipe.df = df_listo

# Re-exportar y obtener resultado usando el Preprocesador (genérico y reutilizable)
df_listo = (pipe
            .exportar('../datos/resultado/df_clean.csv')
            .resultado())

# Explorar el dataset enriquecido con las columnas nuevas
Explorador(df_listo).explorar()

# Verificación final
logger.success(f"Dataset enriquecido — {df_listo.shape[1]} columnas: {df_listo.columns.tolist()}")